# Big Data with PySpark

**What you'll learn:** How to use PySpark to read, filter, aggregate, and query large datasets using both the DataFrame API and SQL.

**Prerequisites:** [01 - Python Basics](01-python-basics.ipynb), [02 - Big Data Intro](02-big-data-intro.ipynb)

**Before you start:** Make sure you've generated the large dataset:
```bash
python scripts/generate_large_data.py
```

## What is Spark?

**Apache Spark** is a distributed computing engine — it's designed to split large tasks across many computers (a "cluster"). But it also works on a single laptop, which is perfect for learning.

**PySpark** is the Python interface to Spark. When you write PySpark code, it gets translated into instructions that the Spark engine executes. The Spark engine runs on the **JVM** (Java Virtual Machine), which is why we installed Java in our environment.

## Starting a Spark Session

Everything in PySpark begins with creating a **SparkSession** — your connection to the Spark engine. In local mode (on your laptop), this starts a small Spark instance right here.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("codingCS") \
    .master("local[*]") \
    .getOrCreate()

print(f"Spark version: {spark.version}")
print("Spark session is ready.")

`local[*]` means "use all available CPU cores on this machine." On a real cluster, you'd replace this with the cluster address.

## Reading Data

Let's read our large sales dataset (100,000 rows):

In [ ]:
df = spark.read.csv("../data/sales_large.csv", header=True, inferSchema=True)

print(f"Rows: {df.count():,}")
print(f"Columns: {len(df.columns)}")

`header=True` tells Spark the first row contains column names. `inferSchema=True` tells it to guess the data types (integer, string, etc.) instead of treating everything as text.

## Inspecting the Data

**printSchema()** shows the column names and types (like pandas `.dtypes`):

In [ ]:
df.printSchema()

**show()** displays the first rows (like pandas `.head()`):

In [ ]:
df.show(5)

## Selecting Columns

Use `.select()` to pick specific columns (like pandas `df[["col1", "col2"]]`):

In [ ]:
df.select("product", "category", "unit_price").show(5)

## Filtering Rows

Use `.filter()` or `.where()` (they do the same thing) to keep only rows that match a condition:

In [ ]:
# Sales in the Electronics category
electronics = df.filter(df["category"] == "Electronics")
print(f"Electronics sales: {electronics.count():,}")
electronics.show(5)

In [ ]:
# Combine conditions: Electronics with quantity > 5
big_electronics = df.filter(
    (df["category"] == "Electronics") & (df["quantity"] > 5)
)
print(f"Big electronics orders: {big_electronics.count():,}")

## Adding New Columns

Use `.withColumn()` to create a new column based on existing ones:

In [ ]:
# Add a total_price column
df_with_total = df.withColumn("total_price", df["quantity"] * df["unit_price"])
df_with_total.select("product", "quantity", "unit_price", "total_price").show(5)

## Grouping and Aggregating

This is where big data tools really shine — summarizing millions of rows.

`.groupBy()` groups rows, then `.agg()` or summary functions compute results per group:

In [ ]:
from pyspark.sql import functions as F

# Total revenue by category
df_with_total = df.withColumn("total_price", df["quantity"] * df["unit_price"])

revenue_by_category = df_with_total.groupBy("category").agg(
    F.sum("total_price").alias("total_revenue"),
    F.count("transaction_id").alias("num_transactions"),
    F.avg("unit_price").alias("avg_price")
)

revenue_by_category.show()

`F.sum()`, `F.count()`, `F.avg()` are Spark's built-in aggregate functions. `.alias()` renames the output column.

## Sorting

Use `.orderBy()` to sort results:

In [ ]:
revenue_by_category.orderBy(F.desc("total_revenue")).show()

## Spark SQL

PySpark also supports **SQL** — the standard language for querying databases. If you know SQL (or want to learn it), this is a powerful option.

First, register your DataFrame as a temporary "table" that SQL can query:

In [ ]:
df.createOrReplaceTempView("sales")

result = spark.sql("""
    SELECT category,
           COUNT(*) AS num_transactions,
           ROUND(AVG(unit_price), 2) AS avg_price,
           SUM(quantity * unit_price) AS total_revenue
    FROM sales
    GROUP BY category
    ORDER BY total_revenue DESC
""")

result.show()

The SQL version produces the same result as the DataFrame API above. Use whichever feels more natural to you.

## Another SQL example

In [ ]:
# Top 10 products by total revenue
spark.sql("""
    SELECT product,
           category,
           SUM(quantity * unit_price) AS total_revenue
    FROM sales
    GROUP BY product, category
    ORDER BY total_revenue DESC
    LIMIT 10
""").show()

## Timing Comparison: PySpark vs pandas

Let's compare how long a groupby operation takes in pandas vs PySpark on the same data:

In [ ]:
import pandas as pd
import time

# pandas version
pdf = pd.read_csv("../data/sales_large.csv")
start = time.time()
pandas_result = pdf.groupby("category").agg(
    total_revenue=("unit_price", lambda x: (x * pdf.loc[x.index, "quantity"]).sum()),
    num_transactions=("transaction_id", "count")
)
pandas_time = time.time() - start

# PySpark version (already loaded)
start = time.time()
spark_result = df.withColumn("total", df["quantity"] * df["unit_price"]) \
    .groupBy("category").agg(
        F.sum("total").alias("total_revenue"),
        F.count("transaction_id").alias("num_transactions")
    ).collect()
spark_time = time.time() - start

print(f"pandas:  {pandas_time:.3f}s")
print(f"PySpark: {spark_time:.3f}s")
print(f"\nNote: at 100K rows, pandas may be faster — Spark's advantage")
print(f"appears at millions of rows where parallelism matters.")

At 100,000 rows, Spark has startup overhead that makes it slower than pandas. Spark's strength is that its time grows much more slowly as data grows — at 10 million or 100 million rows, Spark would pull ahead significantly.

## Writing Results

You can save a Spark DataFrame to CSV (or other formats):

In [ ]:
# Convert Spark result to pandas for easy saving
revenue_pandas = revenue_by_category.toPandas()
print(revenue_pandas)

# You could save it:
# revenue_pandas.to_csv("../data/revenue_summary.csv", index=False)

## Clean Up

Always stop your Spark session when you're done to free up resources:

In [ ]:
spark.stop()
print("Spark session stopped.")

## Key Takeaways

- PySpark is the Python API for Apache Spark, the industry standard for big data.
- You create a `SparkSession` to start, and call `spark.stop()` when done.
- The DataFrame API (`.select()`, `.filter()`, `.groupBy()`) is similar in spirit to pandas.
- Spark SQL lets you write standard SQL queries on your data.
- Spark shines on very large datasets; on small data, pandas is simpler and faster.

---

**Next up:** [04 - Big Data with Dask](04-big-data-dask.ipynb) — a pandas-like alternative for parallel computing.